# Genre2Vec — Data to Embeddings
Reads `songs.csv` and `tags.csv`, trains skip-gram, saves `genre_embeddings.json`.

In [1]:
import numpy as np
import os
import json
from collections import defaultdict
import torch
import torch.nn as nn

## Config

In [ ]:
DATA_DIR = '../data/'
CSV_DIR = DATA_DIR + './csv/'
SONGS_PATH = CSV_DIR + "songs.csv"
OUTPUT_PATH = DATA_DIR + "/embeddings/genre_embeddings.json"

MIN_FREQ = 3
WINDOW = 5
EMBED_DIM = 32
NEG_SAMPLES = 15
LR = 0.001
BATCH_SIZE = 1024

EPOCHS = 200
PATIENCE   = 10
MIN_DELTA  = 0.001

GENRE_BLOCKLIST = {
    "fake", "gbvfi", "russelater", "classify", "commons", "reading", "theme",
    "abstractro", "ambeat", "lift kit", "mandible", "ninja", "escape room",
    "slow game", "corrosion", "fluxwork", "modern downshift",
    "groove room", "composition d", "entehno", "ectofolk",
    "ghoststep", "substep", "tracestep", "zapstep",
    "dubsteppe", "electrofox", "etherpop",
    "axe", "pixie", "lowercase", "scorecore", "epicore", "preverb",
    "lithumania", "metropopolis", "spytrack", "russelater", "mandible",
    "lo star", "channel pop", "bow pop", "soda pop",
    "warm drone", "grave wave", "dark black metal", "black sludge",
    "charred death", "grim death metal", "re:techno", "destroy techno", "christian relaxative",
    "modern uplift", "new tribe",
    "outsider house", "vapor house", "vapor pop", "vapor soul",
    "vapor twitch", "drift", "strut", "solipsynthm", "flick hop",
    "chip hop", "capoeira", "football", "fussball", "wrestling",

    # Children's music in foreign languages (duplicate of children's music)
    "kindermusik", "musica para ninos", "musica per bambini",
    "musiikkia lapsille", "musique pour enfants", "muziek voor kinderen",
    "barnemusikk", "barnmusik",

    # Hyper-local indie scenes
    "albuquerque indie", "austindie", "kc indie", "ok indie", "rva indie",
    "stl indie", "slc indie", "triangle indie", "twin cities indie",
    "louisville indie", "columbus ohio indie", "perth indie", "vegas indie",
    "vienna indie", "vancouver indie", "sheffield indie", "la indie",
    "leeds indie", "athens indie", "dallas indie", "denver indie",
    "brooklyn indie", "boston rock", "portland indie", "seattle indie",
    "bay area indie", "michigan indie", "chicago indie", "northern irish indie",
    "nz indie", "aussietronica", "e6fi", "laboratorio", "bay area hip hop",

    # CCm variants (contemporary christian music — too niche/overlapping)
    "ccm", "deep ccm", "alternative ccm", "german ccm", "brazilian ccm",

    # Misc junk
    "usbm", "tico", "tone", "iskelma", "yoik", "klapa", "enka", "gabba",
    "dansktop", "danspunk", "danseband", "dansband", "folkmusik",
    "levenslied", "schlager", "vintage schlager", "classic schlager",
    "hoerspiel", "kabarett", "liedermacher", "zillertal", "tanzlmusi",
    "blaskapelle", "karneval", "neue deutsche harte", "neue deutsche welle",
    "volksmusik", "ostrock", "wrock", "doujin", "otacore", "vocaloid",
    "anime cv", "anime score", "j-idol", "idol", "thai idol",
    "deep talent show", "talent show", "drama", "comedy", "comic",
    "movie tunes", "french movie tunes", "hollywood", "library music",
    "video game music", "gamecore", "gamelan", "bemani", "c64",
    "demoscene", "chiptune", "deep chiptune", "nintendocore",
}


In [ ]:
paths = [SONGS_PATH]
dirs = [DATA_DIR, CSV_DIR]
for d in dirs:
    if not os.path.exists(d):
        os.makedirs(d)

for p in paths:
    assert os.path.exists(p), f"Missing file: {p}"

os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)

## 1. Load & Clean Data

In [4]:
def read_csv(path):
    with open(path) as f:
        lines = f.read().strip().splitlines()
    headers = [h.strip('"') for h in lines[0].split(',')]
    rows = []
    for line in lines[1:]:
        parts = [p.strip('"') for p in line.split(';')]
        if len(parts) == len(headers):
            rows.append(dict(zip(headers, parts)))
    return rows

docs = defaultdict(set)
for row in read_csv(SONGS_PATH):
    genre = row["genre_name"]
    if genre.lower() not in GENRE_BLOCKLIST:
        docs[row["spotify_id"]].add(genre)

documents = [list(tokens) for tokens in docs.values() if len(tokens) >= 2]
print(f"{len(documents):,} songs | {sum(len(d) for d in documents):,} total tokens")


37,706 songs | 115,274 total tokens


## 2. Vocabulary

In [5]:
freq = defaultdict(int)
for doc in documents:
    for t in doc: freq[t] += 1

idx2token = [t for t, c in sorted(freq.items()) if c >= MIN_FREQ]
token2idx = {t: i for i, t in enumerate(idx2token)}
V = len(idx2token)
print(f"{V} tokens in vocab")

1305 tokens in vocab


## 3. Skip-gram Pairs

In [6]:
pairs = []
for doc in [[token2idx[t] for t in d if t in token2idx] for d in documents]:
    if len(doc) < 2: continue
    for i, c in enumerate(doc):
        for j in range(max(0, i - WINDOW), min(len(doc), i + WINDOW + 1)):
            if j != i: pairs.append((c, doc[j]))

pairs = np.array(pairs, dtype=np.int32)
print(f"{len(pairs):,} training pairs")

333,842 training pairs


## 4. Train Genre2Vec

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

class Genre2Vec(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.W_in  = nn.Embedding(vocab_size, embed_dim)
        self.W_out = nn.Embedding(vocab_size, embed_dim)
        nn.init.uniform_(self.W_in.weight,  -0.1, 0.1)
        nn.init.constant_(self.W_out.weight, 0)

    def forward(self, center, context, negatives):
        vc = self.W_in(center)
        vo = self.W_out(context)
        vn = self.W_out(negatives)
        pos_loss = torch.log(torch.sigmoid((vc * vo).sum(1))).mean()
        neg_loss = torch.log(torch.sigmoid(-(vc.unsqueeze(1) * vn).sum(2))).mean()
        return -(pos_loss + neg_loss)

model = Genre2Vec(V, EMBED_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
pairs_tensor = torch.tensor(pairs, dtype=torch.long)

best_loss = float('inf')
best_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    idx = torch.randperm(len(pairs_tensor))
    pairs_shuffled = pairs_tensor[idx]
    total_loss = 0.0

    for s in range(0, len(pairs_shuffled), BATCH_SIZE):
        batch = pairs_shuffled[s:s+BATCH_SIZE].to(device)
        center    = batch[:, 0]
        context   = batch[:, 1]
        negatives = torch.randint(0, V, (len(batch), NEG_SAMPLES), device=device)

        loss = model(center, context, negatives)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / (len(pairs_shuffled) // BATCH_SIZE)
    print(f"Epoch {epoch+1}/{EPOCHS}:   loss={avg_loss:.4f}")

    if avg_loss < best_loss - MIN_DELTA:
        best_loss = avg_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  no improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(best_state)
Win = model.W_in.weight.detach().cpu().numpy()
print(f"Done. Best loss: {best_loss:.4f}")

Using: cpu
Epoch 1/200:   loss=1.1549
Epoch 2/200:   loss=0.6348
Epoch 3/200:   loss=0.4282
Epoch 4/200:   loss=0.3314
Epoch 5/200:   loss=0.2795
Epoch 6/200:   loss=0.2492
Epoch 7/200:   loss=0.2290
Epoch 8/200:   loss=0.2147
Epoch 9/200:   loss=0.2040
Epoch 10/200:   loss=0.1960
Epoch 11/200:   loss=0.1905
Epoch 12/200:   loss=0.1853
Epoch 13/200:   loss=0.1813
Epoch 14/200:   loss=0.1766
Epoch 15/200:   loss=0.1738
Epoch 16/200:   loss=0.1717
Epoch 17/200:   loss=0.1697
Epoch 18/200:   loss=0.1673
Epoch 19/200:   loss=0.1656
Epoch 20/200:   loss=0.1639
Epoch 21/200:   loss=0.1622
Epoch 22/200:   loss=0.1607
Epoch 23/200:   loss=0.1599
  no improvement (1/10)
Epoch 24/200:   loss=0.1587
Epoch 25/200:   loss=0.1575
Epoch 26/200:   loss=0.1567
  no improvement (1/10)
Epoch 27/200:   loss=0.1556
Epoch 28/200:   loss=0.1546
  no improvement (1/10)
Epoch 29/200:   loss=0.1538
Epoch 30/200:   loss=0.1532
  no improvement (1/10)
Epoch 31/200:   loss=0.1528
Epoch 32/200:   loss=0.1517
Epoch 

## 5. Save Embeddings as JSON

In [8]:
norms = np.linalg.norm(Win, axis=1, keepdims=True) + 1e-9
Win_norm = Win / norms

embeddings = {token: Win_norm[i].tolist() for i, token in enumerate(idx2token)}

with open(OUTPUT_PATH, "w") as f:
    json.dump(embeddings, f)

print(f"Saved {len(embeddings)} embeddings to {OUTPUT_PATH}")

Saved 1305 embeddings to ./data//embeddings/genre_embeddings.json
